In [1]:
import numpy as np, GPy, os,warnings
warnings.filterwarnings("ignore")

ROOT = (
    "C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/FPGA DATA/Raw-data_exahaustive"
)
FPGA = 1
PATHS = range(1, 33)
TOTAL_BUDGET = 0.10
SPLITS = [1, 2, 3, 4, 5, 6, 7, 8, 9]  # AL% out of 10%
K = 2
ROUGH_THRESH = 2.0


def load_dat(fpga, path):
    f = f"{ROOT}/FPGA-{fpga}/data{path}.dat"
    grid = np.loadtxt(f, delimiter=",")
    X, Y = np.where(grid != 0.0)
    vals = grid[X, Y]
    med, std = np.median(vals), np.std(vals)
    mask = np.abs(vals - med) < 3 * std
    return X[mask], Y[mask], vals[mask]


def al_select(X, Y, V, n_al):
    idx = np.random.choice(len(X), 3, replace=False)
    sel = list(idx)
    while len(sel) < n_al:
        Xs = np.c_[X[sel], Y[sel]]
        ys = V[sel].reshape(-1, 1)
        m = GPy.models.GPRegression(Xs, ys, GPy.kern.RBF(2), normalizer=True)
        m.optimize(messages=False)
        rest = [i for i in range(len(X)) if i not in sel]
        Xr = np.c_[X[rest], Y[rest]]
        _, var = m.predict(Xr)
        sel.append(rest[np.argmax(var)])
    return sel


def sde_select(X, Y, V, sel, n_total):
    sel = list(sel)
    rest = [i for i in range(len(X)) if i not in sel]
    np.random.shuffle(rest)
    for i in rest:
        if len(sel) >= n_total:
            break
        ok = True
        for j in sel:
            dx, dy = abs(X[i] - X[j]), abs(Y[i] - Y[j])
            nbr_d = np.mean(np.abs(V[sel] - V[j])) if len(sel) > 1 else 0
            rough = 1 if nbr_d > ROUGH_THRESH else 0
            a, b = 2 + rough, 2 + rough
            if dx < a and dy < b:
                ok = False
                break
        if ok:
            sel.append(i)
    if len(sel) < n_total:
        remain = [i for i in range(len(X)) if i not in sel]
        need = n_total - len(sel)
        sel += list(np.random.choice(remain, need, replace=False))
    return sel


def evaluate(X, Y, V, sel):
    Xs = np.c_[X[sel], Y[sel]]
    ys = V[sel].reshape(-1, 1)
    m = GPy.models.GPRegression(Xs, ys, GPy.kern.RBF(2), normalizer=True)
    m.optimize(messages=False)
    rest = [i for i in range(len(X)) if i not in sel]
    Xr = np.c_[X[rest], Y[rest]]
    pred, _ = m.predict(Xr)
    true = V[rest]
    rmsd = np.sqrt(np.mean((pred.flatten() - true) ** 2))
    dspec = V.max() - V.min()
    delta = np.mean(np.abs(pred.flatten() - true)) / dspec
    return rmsd, delta


print(f"{'AL%':>5} {'SDE%':>5} {'RMSD':>8} {'Delta':>8}")
for al_pct in SPLITS:
    rmsds, deltas = [], []
    for p in PATHS:
        X, Y, V = load_dat(FPGA, p)
        n = len(X)
        n_al = max(3, int(n * al_pct / 100))
        n_total = int(n * TOTAL_BUDGET)
        sel = al_select(X, Y, V, n_al)
        sel = sde_select(X, Y, V, sel, n_total)
        r, d = evaluate(X, Y, V, sel)
        rmsds.append(r)
        deltas.append(d)
    print(
        f"{al_pct:>5} {10 - al_pct:>5} {np.mean(rmsds):>8.4f} {np.mean(deltas):>8.4f}"
    )

  AL%  SDE%     RMSD    Delta
    1     9   1.6598   0.0860
    2     8   1.6588   0.0861
    3     7   1.6613   0.0862
    4     6   1.6703   0.0862
    5     5   1.7230   0.0885
    6     4   1.6789   0.0871
    7     3   1.6990   0.0878
    8     2   1.6800   0.0876
    9     1   1.6865   0.0874


In [2]:
import numpy as np, GPy,warnings
warnings.filterwarnings("ignore")

ROOT = (
    "C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/FPGA DATA/Raw-data_exahaustive"
)
FPGA = 1
PATHS = range(1, 33)
TOTAL_BUDGET = 0.10
SPLITS = [1, 2, 3, 4, 5, 6, 7, 8, 9]  # SVTS% out of 10%
CONFIGS = [(1.0, 0.0), (0.7, 0.3), (0.5, 0.5), (0.3, 0.7), (0.0, 1.0)]
ROUGH_THRESH = 2.0


def load_dat(fpga, path):
    f = f"{ROOT}/FPGA-{fpga}/data{path}.dat"
    grid = np.loadtxt(f, delimiter=",")
    X, Y = np.where(grid != 0.0)
    vals = grid[X, Y]
    med, std = np.median(vals), np.std(vals)
    mask = np.abs(vals - med) < 3 * std
    return X[mask], Y[mask], vals[mask]


def cos_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)


def svts_select(X, Y, V, n_svts, alpha, beta):
    idx = np.random.choice(len(X), 3, replace=False)
    sel = list(idx)
    while len(sel) < n_svts:
        Xs = np.c_[X[sel], Y[sel]]
        ys = V[sel].reshape(-1, 1)
        m = GPy.models.GPRegression(Xs, ys, GPy.kern.RBF(2), normalizer=True)
        m.optimize(messages=False)
        rest = [i for i in range(len(X)) if i not in sel]
        Xr = np.c_[X[rest], Y[rest]]
        mean, var = m.predict(Xr)
        mean, var = mean.flatten(), var.flatten()
        feat = np.c_[
            (mean - mean.mean()) / (mean.std() + 1e-9),
            (var - var.mean()) / (var.std() + 1e-9),
        ]
        sel_feat = feat.mean(axis=0)
        sim = np.array([cos_sim(f, sel_feat) for f in feat])
        sim_n = (sim - sim.min()) / (sim.max() - sim.min() + 1e-9)
        var_n = (var - var.min()) / (var.max() - var.min() + 1e-9)
        score = alpha * sim_n + beta * var_n
        sel.append(rest[np.argmax(score)])
    return sel


def sde_select(X, Y, V, sel, n_total):
    sel = list(sel)
    rest = [i for i in range(len(X)) if i not in sel]
    np.random.shuffle(rest)
    for i in rest:
        if len(sel) >= n_total:
            break
        ok = True
        for j in sel:
            dx, dy = abs(X[i] - X[j]), abs(Y[i] - Y[j])
            nbr_d = np.mean(np.abs(V[sel] - V[j])) if len(sel) > 1 else 0
            rough = 1 if nbr_d > ROUGH_THRESH else 0
            a, b = 2 + rough, 2 + rough
            if dx < a and dy < b:
                ok = False
                break
        if ok:
            sel.append(i)
    if len(sel) < n_total:
        remain = [i for i in range(len(X)) if i not in sel]
        need = n_total - len(sel)
        sel += list(np.random.choice(remain, need, replace=False))
    return sel


def evaluate(X, Y, V, sel):
    Xs = np.c_[X[sel], Y[sel]]
    ys = V[sel].reshape(-1, 1)
    m = GPy.models.GPRegression(Xs, ys, GPy.kern.RBF(2), normalizer=True)
    m.optimize(messages=False)
    rest = [i for i in range(len(X)) if i not in sel]
    Xr = np.c_[X[rest], Y[rest]]
    pred, _ = m.predict(Xr)
    true = V[rest]
    rmsd = np.sqrt(np.mean((pred.flatten() - true) ** 2))
    dspec = V.max() - V.min()
    delta = np.mean(np.abs(pred.flatten() - true)) / dspec
    return rmsd, delta


print(f"{'SVTS%':>6} {'SDE%':>6} {'alpha':>6} {'beta':>6} {'RMSD':>8} {'Delta':>8}")
for svts_pct in SPLITS:
    for alpha, beta in CONFIGS:
        rmsds, deltas = [], []
        for p in PATHS:
            X, Y, V = load_dat(FPGA, p)
            n = len(X)
            n_svts = max(3, int(n * svts_pct / 100))
            n_total = int(n * TOTAL_BUDGET)
            sel = svts_select(X, Y, V, n_svts, alpha, beta)
            sel = sde_select(X, Y, V, sel, n_total)
            r, d = evaluate(X, Y, V, sel)
            rmsds.append(r)
            deltas.append(d)
        print(
            f"{svts_pct:>6} {10 - svts_pct:>6} {alpha:>6} {beta:>6} {np.mean(rmsds):>8.4f} {np.mean(deltas):>8.4f}"
        )


 SVTS%   SDE%  alpha   beta     RMSD    Delta
     1      9    1.0    0.0   1.6876   0.0875
     1      9    0.7    0.3   1.6641   0.0861
     1      9    0.5    0.5   1.6743   0.0866
     1      9    0.3    0.7   1.6594   0.0866
     1      9    0.0    1.0   1.6533   0.0856
     2      8    1.0    0.0   1.6627   0.0866
     2      8    0.7    0.3   1.6637   0.0862
     2      8    0.5    0.5   1.6629   0.0861
     2      8    0.3    0.7   1.6569   0.0861
     2      8    0.0    1.0   1.6562   0.0862
     3      7    1.0    0.0   1.6672   0.0866
     3      7    0.7    0.3   1.6658   0.0861
     3      7    0.5    0.5   1.6512   0.0858
     3      7    0.3    0.7   1.6700   0.0861
     3      7    0.0    1.0   1.6568   0.0861
     4      6    1.0    0.0   1.6916   0.0879
     4      6    0.7    0.3   1.6762   0.0869
     4      6    0.5    0.5   1.6641   0.0863
     4      6    0.3    0.7   1.6901   0.0875
     4      6    0.0    1.0   1.6751   0.0865
     5      5    1.0    0.0   1.70